# Horus · YOLO11 para pistola, humo, llama y paquete — Kaggle · v5b (llamas chicas, afinado bien)

07/11. Con la v4, el encendedor del 06/11 dio **llama 0,37** y los fósforos 0,38: apenas
arriba de 0,35, el umbral con que nace un track. El fuego que vio el modelo hasta ahora
es casi todo D-Fire: incendios de afuera, grandes. Esta versión suma llamas chicas de
casa (celda 4b, `fuego_ext.json`): velas, encendedores y fósforos de cerca, fuegos dentro
de casas vistos por cámaras de seguridad, y luces y lámparas anotadas como "no es fuego".
Se **afina la v4** (sus pesos como punto de partida, 40 épocas) con todo junto: lo de
siempre, las armas de la v4 y el fuego nuevo.

**v5b.** La primera corrida (versión 5) se cortó en la época 21: con el optimizador
automático (MuSGD a 0,01, que además ignora `--lr0`) y el warmup de siempre, el val se
cayó de 0,67 a 0,51 y la paciencia de 20 dejó como best.pt la época 1, casi la v4. Ahora:
**SGD a 0,003, una época de warmup sin el bias alto, sin corte temprano**, y best.pt se
elige con el val de mezcla_v1 + armas + fuego (antes solo mezcla_v1, que no tiene velas).
También se guarda y se mide la última época (`horus_yolo4_ultimo.pt`).

**Antes de Save Version → Run All, en el panel de la derecha:**
1. **Accelerator → GPU T4 x2.**
2. **Internet → On** (Open Images y Pyro-SDIS se bajan desde acá).
3. **Inputs:** D-Fire (`sayedgamal99/smoke-fire-detection-yolo`), `teonieto/horus-propio`,
   la salida de `horus-yolo4` **versión 4** (la v4: punto de partida y comparación), los
   datasets de armas de la v4 y estos:
   - `sreemantabarman/flame-dataset-candlelightermatch-stick-flames` (train)
   - `pengbo00/home-fire-dataset` (train)
   - `ironwolf437/fire-detection-dataset` (train)
   - `sinchanashivanand/indoor-fire-and-smoke-detection-with-yolov8` (prueba)

Al final, en **Output → modelos_yolo/** quedan `horus_yolo4.pt`, `horus_yolo4_416.onnx`
y `metricas.json`: v5 contra v4 en el val de siempre, en val y prueba de armas y en val y
prueba de fuego, con la medida por foto de las cuatro clases.

## 1 · Entorno

In [ ]:
import os, sys, subprocess, shutil, pathlib, json
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "SIN GPU")
import torch
if not torch.cuda.is_available():
    raise SystemExit("Sin GPU. Panel derecho -> Accelerator -> GPU, y volvé a correr.")
NGPU = torch.cuda.device_count()
print("torch", torch.__version__, "· GPUs:", NGPU)
TRABAJO = "/kaggle/working"
ENTRADA = pathlib.Path("/kaggle/input")
print("entradas adjuntas:")
for p in sorted(ENTRADA.glob("datasets/*/*")) + sorted(ENTRADA.glob("notebooks/*/*")) + \
         sorted(p for p in ENTRADA.glob("*") if p.name not in ("datasets", "notebooks")):
    print("  ", p)

## 2 · Internet, el repo y ultralytics

In [ ]:
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("Sin internet. Panel derecho -> Internet -> On.")
REPO, RAMA = "https://github.com/Teo50000/Horus-AI.git", "Models"
RAIZ_REPO = "/kaggle/temp/Horus-AI"
if os.path.exists(RAIZ_REPO): shutil.rmtree(RAIZ_REPO)
subprocess.run(["git", "clone", "--depth", "1", "-b", RAMA, REPO, RAIZ_REPO], check=True)
OBJ = f"{RAIZ_REPO}/horus/04_cabezas/objetos"
print("OBJ =", OBJ)
%pip install -q ultralytics fiftyone datasets pycocotools 2>&1 | tail -2

## 3 · D-Fire y lo propio, por nombre

06/11: con los datasets de armas adjuntos (varios también en formato YOLO) ya no sirve
"el primero que parezca YOLO": se habría llevado un dataset de armas como si fuera
humo y llama. D-Fire se busca por su nombre y el propio por su manifiesto, solo entre
los datasets de teonieto (recorrer todo /kaggle/input con 200.000 archivos tarda).

07/11: la salida adjunta de `horus-yolo4` tiene que ser la de la **versión 4**. Sus pesos
son el punto de partida (se afina la v4) y la vara para comparar. Se verifica el sha.

In [ ]:
DFIRE = ("smoke-fire-detection-yolo", "d-fire", "dfire")
cands = [d for d in sorted(ENTRADA.glob("datasets/*/*")) + sorted(ENTRADA.glob("*"))
         if any(s in d.name.lower() for s in DFIRE)]
if not cands:
    raise SystemExit("No encuentro D-Fire adjunto (sayedgamal99/smoke-fire-detection-yolo).")
origen = cands[0]
crudo = pathlib.Path(OBJ) / "datasets" / "_crudo" / "d-fire"
crudo.parent.mkdir(parents=True, exist_ok=True)
if crudo.exists() or crudo.is_symlink(): crudo.unlink()
crudo.symlink_to(origen, target_is_directory=True)
print(f"✓ D-Fire enlazado desde {origen}")
mans = sorted((ENTRADA / "datasets" / "teonieto").rglob("horus_propio.json")) \
    if (ENTRADA / "datasets" / "teonieto").exists() else []
PROPIO = str(mans[0].parent) if mans else "no"
print("dataset propio:", PROPIO)
# La salida de horus-yolo4 v4 (adjunta como input): en notebooks/<dueño>/<slug>/ o,
# con el montaje viejo, en /kaggle/input/<slug>/. Sin recorrer los datasets.
import hashlib
V4_SHA = "af21c236a029387a2f5400df8c12c5a867bdb015f9c6ed41d23d8f7603e81f14"
ANTERIOR = None
for base in [ENTRADA / "notebooks"] + [p for p in ENTRADA.glob("*") if p.name not in ("datasets", "notebooks")]:
    if base.is_dir():
        ANTERIOR = next(iter(sorted(base.rglob("modelos_yolo/horus_yolo4.pt"))), None)
        if ANTERIOR:
            break
if ANTERIOR:
    sha = hashlib.sha256(ANTERIOR.read_bytes()).hexdigest()
    print("anterior:", ANTERIOR, "· sha", sha[:12], "· ES LA V4" if sha == V4_SHA else "· ⚠ NO es la v4")
else:
    print("⚠ no está la salida de horus-yolo4: se entrena desde COCO, 80 épocas, sin comparar")

## 4 · Bajar el resto y armar la mezcla (lo mismo que para la cabeza propia)

In [ ]:
%cd {OBJ}
!python bajar_datasets.py --verificar
!python bajar_datasets.py --descargar pyro-sdis openimages
!python bajar_datasets.py --normalizar
!python bajar_datasets.py --armar

## 4b · Armas y llamas chicas: datasets públicos (`armas_ext.py`)

Normaliza datasets públicos al formato de `mezcla_v1` (7 clases) en tres partes:
**train** entra entero al YOLO4 (sin tope de negativos: los de acá son los difíciles),
**val** se mide aparte, y **prueba** son fuentes que el modelo no ve nunca. Lo de val y
prueba que se parezca (dHash, Hamming ≤ 5, también en espejo) a algo de train o del val
de `mezcla_v1` se saca: los datasets de Kaggle son remezclas unos de otros.

Dos carpetas: `armas_ext` (la misma de la v4: mismo val y misma prueba) y `fuego_ext`
(07/11): velas, encendedores y fósforos, fuegos de casa, y luces y lámparas anotadas como
negativos difíciles, que entran sin tope.

In [ ]:
%%writefile /kaggle/temp/Horus-AI/horus/04_cabezas/objetos/armas_ext.py
# -*- coding: utf-8 -*-
"""
armas_ext.py · Datasets públicos de armas para el YOLO de 4 clases (06/11).

Por qué
-------
El 06/11 a las 12:23 la v3 marcó "pistola" (0,65, sostenida 9 s) en una media que
Teo tenía en la mano. La pistola de la v3 sale solo de Open Images: 490 cajas de
fotos de Flickr, armas grandes y centradas, y ni un negativo del tipo "mano
sosteniendo algo que NO es un arma". Este script trae datasets públicos con las
dos cosas: pistolas en contextos variados (cámaras, videos, calle) y manos con
celulares, billeteras, tarjetas, billetes y cuchillos.

Qué arma
--------
Una carpeta con el formato de mezcla_v1 (7 clases: humo llama persona pistola
cuchillo celular paquete), en tres partes:

  images|labels/train    entra ENTERA al YOLO4 (positivos y negativos, sin tope)
  images|labels/val      val aparte de las fuentes nuevas; no se mezcla con el de
                         mezcla_v1, así la métrica vieja sigue siendo comparable
  images|labels/prueba   fuentes que el modelo no ve nunca: dicen si generaliza

Reglas por fuente (lista "fuentes" del JSON de --config)
-------------------------------------------------------
  alias     prefijo de los archivos (corto, sin espacios)
  ruta      carpeta bajo /kaggle/input, o absoluta
  mapa      clase del dataset -> clase nuestra. Lo que no está ni en mapa, ni en
            excluir, ni en ignorar deja la foto AFUERA (y se avisa): mejor perder
            una foto que enseñarle algo mal.
  excluir   si la foto tiene una de estas clases no entra. Los rifles van acá: la
            clase es "pistola" y no queremos enseñarle ni que un rifle es pistola
            ni que es fondo.
  ignorar   la caja se tira pero la foto entra ("person" en un dataset que no
            anota a todas las personas: al YOLO4 la persona no le importa).
  vacias    "negativo" si el dataset trae fotos sin cajas A PROPÓSITO (carpeta
            no_weapon); "descartar" si no (suelen ser fotos que nadie anotó).
  rol       "train" (90/10 en train/val) o "prueba".
  video     true si son cuadros de video: el split va por video y no por cuadro,
            para que el val no tenga el cuadro de al lado de uno de train.
  tope_pos, tope_neg   máximo de fotos con pistola / sin pistola de esa fuente.
  nombres   opcional: lista o dict de nombres YOLO si el data.yaml no sirve.
  bloque    (07/11) con video: agrupa por número de cuadro de a `bloque`
            ("frame_0391" -> grupo frame#13 con bloque 30). Para datasets donde
            todos los cuadros se llaman frame_NNNN: sin esto serían UN grupo y
            caerían todos del mismo lado del split.
  siempre   (07/11) clases que se ignoran pero cuya foto entra SIN tope: los
            negativos difíciles anotados a propósito (lámparas, luces, cosas
            naranjas que no son fuego).
  lado_min  (07/11) lado relativo mínimo (raíz de ancho x alto) de una caja de
            pistola, llama o humo; si hay una más chica, la foto no entra. Una
            llama de 3 px a 416 es un punto brillante: enseñarle que eso es
            fuego es enseñarle a ver fuego en los LEDs del cuarto.

07/11 · fuego_ext.json usa este mismo script para llamas chicas de casa (velas,
encendedores, fósforos, fuegos de interior). Una caja de llama o humo demasiado
chica deja la foto afuera, igual que una pistola: tirarla dejaría un fuego sin
anotar. Con las fuentes de armas el resultado es idéntico al de la v4 (mismo val
y misma prueba de armas).

Duplicados: dHash de 64 bits (y el del espejo). Lo de val y prueba que esté a
distancia <= 5 de algo de train, o de algo del val de mezcla_v1, se saca: los
datasets de armas de Kaggle son remezclas unos de otros y sin esto la prueba
"de afuera" mediría memoria.

    python armas_ext.py --config armas_ext.json --salida datasets/armas_ext \\
        --mezcla datasets/mezcla_v1
    python armas_ext.py --prueba     # datos sintéticos, verifica el camino entero
"""

from __future__ import annotations

import argparse
import collections
import json
import os
import random
import re
import shutil
import sys
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

CLASES_7 = ("humo", "llama", "persona", "pistola", "cuchillo", "celular", "paquete")
IDX = {c: i for i, c in enumerate(CLASES_7)}
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PISTOLA = IDX["pistola"]
# Clases que no se pueden tirar si la caja es mínima: la foto se va entera.
MINIMA_EXCLUYE = {PISTOLA, IDX["llama"], IDX["humo"]}
COLOR = {PISTOLA: (255, 40, 40), IDX["llama"]: (255, 150, 0), IDX["humo"]: (210, 210, 210)}


def norm(n) -> str:
    return re.sub(r"[\s_\-]+", " ", str(n).strip().lower())


# --------------------------------------------------------------------------- #
# Lectura de una fuente
# --------------------------------------------------------------------------- #
# /kaggle/input es un montaje de red: cada exists()/stat()/open() es un pedido.
# Por eso: un solo os.walk por fuente, las etiquetas se buscan en la lista ya
# hecha (sin stat) y se leen en paralelo. El relevamiento del 06/11 tardaba más
# de 10 min por dataset haciendo exists() por foto; así, 8 s.
def _listar(raiz: Path) -> List[str]:
    todos = []
    for r, _, fs in os.walk(raiz, followlinks=True):
        todos += [os.path.join(r, x) for x in fs]
    return todos


def _leer(p: str) -> Optional[str]:
    try:
        with open(p, errors="ignore") as fh:
            return fh.read()
    except Exception:
        return None


def _en_paralelo(fn, xs, hilos: int = 32):
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(hilos) as ex:
        return list(ex.map(fn, xs))


def _nombres_yolo(todos: List[str], forzados=None) -> Dict[int, str]:
    if forzados:
        if isinstance(forzados, dict):
            return {int(k): norm(v) for k, v in forzados.items()}
        return {i: norm(v) for i, v in enumerate(forzados)}
    try:
        import yaml
    except ImportError:
        yaml = None
    if yaml is not None:
        for y in sorted(a for a in todos if a.endswith((".yaml", ".yml"))):
            try:
                d = yaml.safe_load(_leer(y) or "")
            except Exception:
                continue
            n = d.get("names") if isinstance(d, dict) else None
            if isinstance(n, dict) and n:
                return {int(k): norm(v) for k, v in n.items()}
            if isinstance(n, list) and n:
                return {i: norm(v) for i, v in enumerate(n)}
    for c in sorted(a for a in todos if a.endswith(("classes.txt", ".names"))):
        ls = [l.strip() for l in (_leer(c) or "").splitlines() if l.strip()]
        if ls:
            return {i: norm(v) for i, v in enumerate(ls)}
    return {}


def _etiqueta_yolo(img: str, txts: set) -> Optional[str]:
    b = os.path.splitext(img)[0]
    if b + ".txt" in txts:
        return b + ".txt"
    partes = img.split(os.sep)
    if "images" in partes:
        i = len(partes) - 1 - partes[::-1].index("images")
        q = partes.copy()
        q[i] = "labels"
        c = os.path.splitext(os.sep.join(q))[0] + ".txt"
        if c in txts:
            return c
    return None


def _cajas_yolo(texto: str, nombres: Dict[int, str]) -> List[Tuple[str, float, float, float, float]]:
    out = []
    for linea in texto.splitlines():
        t = linea.split()
        if len(t) < 5:
            continue
        try:
            c = int(float(t[0]))
            v = [float(x) for x in t[1:]]
        except ValueError:
            continue
        if len(v) == 4:
            x, y, w, h = v
        elif len(v) >= 6 and len(v) % 2 == 0:          # polígono de segmentación
            xs, ys = v[0::2], v[1::2]
            x, y = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
            w, h = max(xs) - min(xs), max(ys) - min(ys)
        else:
            continue
        out.append((nombres.get(c, f"#{c}"), x, y, w, h))
    return out


def _cajas_voc(texto: str) -> List[Tuple[str, float, float, float, float]]:
    try:
        t = ET.fromstring(texto)
    except Exception:
        return []
    sw = float(t.findtext("size/width") or 0)
    sh = float(t.findtext("size/height") or 0)
    if not sw or not sh:
        return []
    out = []
    for o in t.findall("object"):
        b = o.find("bndbox")
        if b is None:
            continue
        x1, y1, x2, y2 = (float(b.findtext(k) or 0) for k in ("xmin", "ymin", "xmax", "ymax"))
        out.append((norm(o.findtext("name") or "?"), (x1 + x2) / 2 / sw, (y1 + y2) / 2 / sh,
                    (x2 - x1) / sw, (y2 - y1) / sh))
    return out


def recorrer(raiz: Path, f: Optional[dict] = None) -> List[Tuple[Path, Optional[list]]]:
    """(imagen, cajas) de una fuente. cajas=None: la foto no tiene etiqueta.

    f["formato"]: "auto" (YOLO, si no VOC, si no COCO), "yolo", "voc" o "coco".
    f["filtro"]: regex sobre la ruta relativa; solo entran las fotos que la cumplen.
    """
    f = f or {}
    todos = _listar(raiz)
    filtro = re.compile(f["filtro"]) if f.get("filtro") else None
    rel = lambda a: os.path.relpath(a, raiz).replace(os.sep, "/")
    imgs = sorted(a for a in todos if os.path.splitext(a)[1].lower() in EXT
                  and (filtro is None or filtro.search(rel(a))))
    set_imgs = set(imgs)
    formato = f.get("formato", "auto")
    out: Dict[str, list] = {}
    if formato in ("auto", "yolo"):
        txts = {a for a in todos if a.endswith(".txt")}
        nombres = _nombres_yolo(todos, f.get("nombres"))
        pares = [(im, _etiqueta_yolo(im, txts)) for im in imgs]
        pares = [(im, lb) for im, lb in pares if lb]
        for (im, _), tx in zip(pares, _en_paralelo(_leer, [lb for _, lb in pares])):
            if tx is not None:
                out[im] = _cajas_yolo(tx, nombres)
    if formato in ("auto", "voc"):
        xmls = {Path(a).stem: a for a in todos if a.lower().endswith(".xml")}
        faltan = [im for im in imgs if im not in out and Path(im).stem in xmls]
        for im, tx in zip(faltan, _en_paralelo(_leer, [xmls[Path(im).stem] for im in faltan])):
            if tx is not None:
                out[im] = _cajas_voc(tx)
    if formato in ("auto", "coco"):
        por_nombre: Dict[str, List[str]] = collections.defaultdict(list)
        for im in imgs:
            por_nombre[os.path.basename(im)].append(im)
        for j in sorted(a for a in todos if a.lower().endswith(".json")
                        and (filtro is None or filtro.search(rel(a)))):
            try:
                d = json.loads(_leer(j) or "")
            except Exception:
                continue
            if not (isinstance(d, dict) and {"images", "annotations", "categories"} <= set(d)):
                continue
            cats = {c["id"]: norm(c["name"]) for c in d["categories"]}
            anns = collections.defaultdict(list)
            for a in d["annotations"]:
                anns[a.get("image_id")].append(a)
            base_j = os.path.dirname(j)
            for info in d["images"]:
                fn = str(info.get("file_name", ""))
                cand = os.path.normpath(os.path.join(base_j, fn))
                if cand not in set_imgs:
                    # por nombre, pero solo cerca del json (su carpeta o la de al lado):
                    # dos splits pueden tener fotos con el mismo nombre
                    arriba = os.path.dirname(base_j) + os.sep
                    lst = [x for x in por_nombre.get(os.path.basename(fn), []) if x.startswith(arriba)]
                    mismos = [x for x in lst if os.path.dirname(x) == base_j]
                    cand = mismos[0] if mismos else (lst[0] if len(lst) == 1 else None)
                if cand is None or cand in out:
                    continue
                W, H = float(info.get("width") or 0), float(info.get("height") or 0)
                if not W or not H:
                    continue
                bx = []
                for a in anns.get(info.get("id"), []):
                    if not a.get("bbox"):
                        continue
                    x, y, w, h = (float(v) for v in a["bbox"][:4])
                    bx.append((cats.get(a.get("category_id"), f"#{a.get('category_id')}"),
                               (x + w / 2) / W, (y + h / 2) / H, w / W, h / H))
                out[cand] = bx
    return [(Path(im), out.get(im)) for im in imgs]


# --------------------------------------------------------------------------- #
# Hash perceptual (dHash 64 bits) e índice para buscar parecidos
# --------------------------------------------------------------------------- #
def _hash_pil(I) -> Tuple[int, int]:
    px = list(I.convert("L").resize((9, 8)).tobytes())
    h = hf = 0
    for fila in range(8):
        f = px[fila * 9:(fila + 1) * 9]
        for col in range(8):
            h = (h << 1) | (f[col] > f[col + 1])
        r = f[::-1]
        for col in range(8):
            hf = (hf << 1) | (r[col] > r[col + 1])
    return h, hf


def dhash(p: Path) -> Optional[Tuple[int, int]]:
    from PIL import Image
    try:
        with Image.open(p) as I:
            I.draft("L", (64, 64))
            return _hash_pil(I)
    except Exception:
        return None


def _copiar(args) -> Tuple[int, Optional[Tuple[int, int]]]:
    """Lee la foto UNA vez: la copia achicada (lado mayor <= `lado`, EXIF aplicado,
    JPEG) al disco local y devuelve su dHash. Leer de /kaggle/input en cada época
    es lo lento; además una foto de 4000 px se decodifica 80 veces para usarla a 416.
    """
    k, origen, destino, lado = args
    from PIL import Image, ImageOps
    try:
        with Image.open(origen) as I:
            I.draft("RGB", (lado, lado))
            I = ImageOps.exif_transpose(I).convert("RGB")
            w, h = I.size
            if max(w, h) > lado:
                f = lado / max(w, h)
                I = I.resize((max(1, round(w * f)), max(1, round(h * f))), Image.BILINEAR)
            I.save(destino, "JPEG", quality=90)
            return k, _hash_pil(I)
    except Exception:
        return k, None


class Indice:
    """Distancia de Hamming <= tol contra todo lo agregado (4 tablas de 16 bits)."""

    def __init__(self, tol: int = 5):
        self.tol = tol
        self.t = [collections.defaultdict(list) for _ in range(4)]

    def agregar(self, h: int) -> None:
        for k in range(4):
            self.t[k][(h >> (16 * k)) & 0xFFFF].append(h)

    def parecido(self, h: int) -> bool:
        for k in range(4):
            for o in self.t[k].get((h >> (16 * k)) & 0xFFFF, ()):
                if bin(o ^ h).count("1") <= self.tol:
                    return True
        return False


# --------------------------------------------------------------------------- #
def _grupo_video(p: Path, bloque: Optional[int] = None) -> str:
    """'video3_frame_0012' -> 'video3_frame': los cuadros de un video, juntos.

    Solo si lo que queda todavía identifica un video (tiene un número o dice
    frame/video/clip); 'nw123' sigue siendo su propio grupo, si no todos los
    negativos de una carpeta caerían del mismo lado.

    Con `bloque`: 'frame_0391' -> 'frame#13' (391 // 30). Las copias aumentadas
    de Roboflow (".rf.<hash>") de un mismo cuadro caen siempre juntas.
    """
    s = p.stem.split(".rf.")[0]
    s = re.sub(r"_(jpg|jpeg|png)$", "", s, flags=re.I)
    if bloque:
        m = re.search(r"(\d+)$", s)
        if m:
            return f"{p.parent.name}/{s[:m.start()].rstrip('_-. ')}#{int(m.group(1)) // bloque}"
        return f"{p.parent.name}/{s}"
    pref = re.sub(r"[\W_]*\d+$", "", s)
    if pref != s and (re.search(r"\d", pref) or re.search(r"frame|video|vid|clip|cam", pref, re.I)):
        s = pref
    return str(p.parent.name) + "/" + s


def _al_val(clave: str, frac: float) -> bool:
    import hashlib
    return int(hashlib.md5(clave.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF < frac


def clasificar(cajas: Optional[list], f: dict, avisos: collections.Counter):
    """-> ("pos"|"neg"|None, filas 7 clases, motivo)"""
    if cajas is None:
        return (("neg", [], "") if f.get("vacias") == "negativo" else (None, [], "sin_etiqueta"))
    mapa = {norm(k): v for k, v in f.get("mapa", {}).items()}
    excluir = {norm(x) for x in f.get("excluir", [])}
    ignorar = {norm(x) for x in f.get("ignorar", [])}
    filas = []
    for n, x, y, w, h in cajas:
        n = norm(n)
        if n in excluir:
            return None, [], f"excluye:{n}"
        if n in ignorar or ("*" in ignorar and n not in mapa):
            continue
        if n not in mapa:
            avisos[n] += 1
            return None, [], f"desconocida:{n}"
        x1, y1 = max(0.0, x - w / 2), max(0.0, y - h / 2)
        x2, y2 = min(1.0, x + w / 2), min(1.0, y + h / 2)
        if x2 - x1 < 0.003 or y2 - y1 < 0.003:
            # Una pistola de menos de un píxel a 416 no se puede aprender, pero
            # tampoco se puede tirar la caja y dejar la foto como negativo.
            # Lo mismo una llama o un humo (07/11).
            if IDX[mapa[n]] in MINIMA_EXCLUYE:
                return None, [], f"{mapa[n]}_minima"
            continue
        if (f.get("lado_min") and IDX[mapa[n]] in MINIMA_EXCLUYE
                and ((x2 - x1) * (y2 - y1)) ** 0.5 < f["lado_min"]):
            return None, [], f"{mapa[n]}_chica"
        filas.append(f"{IDX[mapa[n]]} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} "
                     f"{x2 - x1:.6f} {y2 - y1:.6f}")
    if not filas and not cajas:
        return (("neg", [], "") if f.get("vacias") == "negativo" else (None, [], "vacia"))
    if any(int(r.split()[0]) == PISTOLA for r in filas):
        return "pos", filas, ""
    return "neg", filas, ""


def _hoja(destino: Path, fotos: list) -> None:
    """12 fotos con sus cajas, JPEG en base64 de menos de 100 KB: el visor de
    Kaggle muestra hasta ahí. Cada foto es (img, filas) o (img, filas, crudas):
    las crudas son cajas ignoradas, en amarillo con el nombre original.
    Pistola en rojo, llama naranja, humo gris claro, el resto azul."""
    import base64
    import io
    from PIL import Image, ImageDraw
    hoja = Image.new("RGB", (4 * 240, 3 * 180), (20, 20, 20))
    for k, item in enumerate(fotos[:12]):
        img, filas = item[0], item[1]
        crudas = item[2] if len(item) > 2 else []
        try:
            with Image.open(img) as I:
                I.draft("RGB", (480, 360))
                I = I.convert("RGB").resize((240, 180))
        except Exception:
            continue
        d = ImageDraw.Draw(I)
        cajas = [(CLASES_7[int(c)], COLOR.get(int(c), (60, 140, 255)), float(cx), float(cy), float(w), float(h))
                 for c, cx, cy, w, h in (x.split() for x in filas)]
        cajas += [(str(n)[:12], (255, 255, 0), x, y, w, h) for n, x, y, w, h in crudas]
        for nom, col, cx, cy, w, h in cajas:
            d.rectangle([(cx - w / 2) * 240, (cy - h / 2) * 180, (cx + w / 2) * 240, (cy + h / 2) * 180],
                        outline=col, width=2)
            d.text(((cx - w / 2) * 240 + 2, max(0, (cy - h / 2) * 180 - 10)), nom, fill=col)
        hoja.paste(I, ((k % 4) * 240, (k // 4) * 180))
    destino.parent.mkdir(parents=True, exist_ok=True)
    for q in (50, 40, 30):
        b = io.BytesIO()
        hoja.save(b, "JPEG", quality=q)
        if len(b.getvalue()) * 4 / 3 < 100_000:
            break
    destino.write_text(base64.b64encode(b.getvalue()).decode())


def armar(config: dict, salida: Path, mezcla: Optional[Path], base: Path,
          frac_val: float = 0.10, semilla: int = 0, tol: int = 5,
          lado: int = 1024, procesos: Optional[int] = None, revisar: bool = False) -> dict:
    """revisar=True: solo clasifica y cuenta, y deja una hoja de muestras por fuente
    (pistola en rojo, el resto en azul) en salida/revision/. No copia nada."""
    from multiprocessing import Pool
    rnd = random.Random(semilla)
    if salida.exists():
        shutil.rmtree(salida)
    for parte in ("train", "val", "prueba"):
        (salida / "images" / parte).mkdir(parents=True, exist_ok=True)
        (salida / "labels" / parte).mkdir(parents=True, exist_ok=True)
    tmp = salida / "_copias"
    tmp.mkdir()

    # 1) el val de mezcla_v1: nada nuevo puede parecerse a eso
    idx_mezcla_val = Indice(tol)
    if mezcla is not None and not revisar:
        for d in (mezcla / "images" / "val", mezcla / "val" / "images"):
            if d.is_dir():
                for p in d.iterdir():
                    if p.suffix.lower() in EXT:
                        h = dhash(p)
                        if h:
                            idx_mezcla_val.agregar(h[0])
                print(f"[ext] val de mezcla_v1 indexado: {d}", flush=True)
                break

    resumen = {}
    candidatas = []        # (fuente, img, filas)
    fuentes = config["fuentes"]
    # las de train primero: lo de prueba se compara contra todo lo de train
    fuentes = sorted(fuentes, key=lambda f: f.get("rol", "train") != "train")
    for f in fuentes:
        raiz = Path(f["ruta"])
        if not raiz.is_absolute():
            raiz = base / raiz
        r = collections.Counter()
        avisos = collections.Counter()
        if not raiz.exists():
            print(f"[ext] ⚠ {f['alias']}: no existe {raiz}; la salteo", flush=True)
            resumen[f["alias"]] = {"error": f"no existe {raiz}"}
            continue
        pos, neg = [], []
        siempre = {norm(x) for x in f.get("siempre", [])}
        ign_set = {norm(x) for x in f.get("ignorar", [])}
        mapa_set = {norm(x) for x in f.get("mapa", {})}
        fijas = []                  # (img, filas, tipo): negativos difíciles, sin tope
        con_ignoradas, vacias = [], []   # solo para las hojas de --revisar
        for img, cajas in recorrer(raiz, f):
            r["fotos"] += 1
            if revisar and cajas is not None:
                if not cajas:
                    vacias.append((img, []))
                else:
                    ign = [c for c in cajas if norm(c[0]) in ign_set
                           or ("*" in ign_set and norm(c[0]) not in mapa_set)]
                    if ign:
                        con_ignoradas.append((img, None, ign))
            tipo, filas, motivo = clasificar(cajas, f, avisos)
            if tipo is None:
                r[motivo.split(":")[0]] += 1
                if motivo.startswith("excluye"):
                    r[motivo] += 1
                continue
            if siempre and cajas and any(norm(c[0]) in siempre for c in cajas):
                fijas.append((img, filas, tipo))
                continue
            (pos if tipo == "pos" else neg).append((img, filas))
        rnd.shuffle(pos)
        rnd.shuffle(neg)
        pos = pos[: f.get("tope_pos", len(pos))]
        neg = neg[: f.get("tope_neg", len(neg))]
        if fijas:
            r["fijas_sin_tope"] = len(fijas)
            pos += [(i, fl) for i, fl, t in fijas if t == "pos"]
            neg += [(i, fl) for i, fl, t in fijas if t == "neg"]
        candidatas += [(f, img, filas) for img, filas in pos + neg]
        r["pos_elegidas"], r["neg_elegidas"] = len(pos), len(neg)
        if avisos:
            r["_clases_desconocidas"] = dict(avisos.most_common(12))
        cl = collections.Counter(CLASES_7[int(x.split()[0])] for _, fs in pos + neg for x in fs)
        r["_cajas_elegidas"] = dict(cl)
        lados = sorted((float(x.split()[3]) * float(x.split()[4])) ** 0.5
                       for _, fs in pos for x in fs if int(x.split()[0]) == PISTOLA)
        if lados:
            r["_pistola_lado_mediano"] = round(lados[len(lados) // 2], 3)
        for nc in ("llama", "humo"):
            ls = sorted((float(x.split()[3]) * float(x.split()[4])) ** 0.5
                        for _, fs in pos + neg for x in fs if int(x.split()[0]) == IDX[nc])
            if ls:
                r[f"_{nc}_lado_mediano"] = round(ls[len(ls) // 2], 3)
                r[f"_{nc}_chicas_<0.05"] = round(sum(v < 0.05 for v in ls) / len(ls), 3)
        con = sum(1 for _, fs in pos + neg if fs)
        r["fotos_con_cajas"], r["fotos_sin_cajas"] = con, len(pos) + len(neg) - con
        resumen[f["alias"]] = r
        print(f"[ext] {f['alias']}: " + json.dumps(dict(r), ensure_ascii=False), flush=True)
        if revisar:
            if pos:
                muestra = rnd.sample(pos, min(8, len(pos))) + rnd.sample(neg, min(4, len(neg)))
            else:
                # sin pistolas (fuego): 9 con cajas y 3 sin, si hay
                con_c = [x for x in neg if x[1]]
                sin_c = [x for x in neg if not x[1]]
                k_sin = min(3, len(sin_c))
                muestra = rnd.sample(con_c, min(12 - k_sin, len(con_c))) + rnd.sample(sin_c, k_sin)
            _hoja(salida / "revision" / f"hoja_{f['alias']}.txt", muestra)
            if con_ignoradas:
                _hoja(salida / "revision" / f"hoja_{f['alias']}_ignoradas.txt",
                      [(i, [], ig) for i, _, ig in rnd.sample(con_ignoradas, min(12, len(con_ignoradas)))])
                r["revision_con_ignoradas"] = len(con_ignoradas)
            if vacias:
                _hoja(salida / "revision" / f"hoja_{f['alias']}_vacias.txt",
                      rnd.sample(vacias, min(12, len(vacias))))
                r["revision_etiqueta_vacia"] = len(vacias)

    if revisar:
        for alias in list(resumen):
            resumen[alias] = dict(resumen[alias])
        (salida / "resumen.json").write_text(json.dumps(resumen, indent=1, ensure_ascii=False))
        return resumen

    # copias locales + hash, en paralelo
    trabajos = [(k, str(img), str(tmp / f"{k:07d}.jpg"), lado)
                for k, (_, img, _) in enumerate(candidatas)]
    hashes: Dict[int, Optional[Tuple[int, int]]] = {}
    with Pool(procesos or os.cpu_count() or 2) as pool:
        for i, (k, h) in enumerate(pool.imap_unordered(_copiar, trabajos, chunksize=16)):
            hashes[k] = h
            if (i + 1) % 2000 == 0:
                print(f"[ext] copiadas {i + 1}/{len(trabajos)}", flush=True)

    elegidas = []          # (fuente, rol_final, copia, filas, hash)
    vistos_exactos = set()
    for k, (f, img, filas) in enumerate(candidatas):
        r = resumen[f["alias"]]
        h = hashes.get(k)
        copia = tmp / f"{k:07d}.jpg"
        if h is None:
            r["ilegible"] += 1
            continue
        if h[0] in vistos_exactos:
            r["duplicada"] += 1
            continue
        if idx_mezcla_val.parecido(h[0]) or idx_mezcla_val.parecido(h[1]):
            r["igual_a_val_mezcla"] += 1
            continue
        vistos_exactos.add(h[0])
        rol = f.get("rol", "train")
        if rol == "train":
            clave = _grupo_video(img, f.get("bloque")) if f.get("video") else str(img)
            rol = "val" if _al_val(f["alias"] + "|" + clave, frac_val) else "train"
        elegidas.append((f["alias"], rol, copia, filas, h))
    for alias in list(resumen):
        resumen[alias] = dict(resumen[alias])

    # 2) val y prueba no pueden parecerse a nada de train
    idx_train = Indice(tol)
    for _, rol, _, _, h in elegidas:
        if rol == "train":
            idx_train.agregar(h[0])
            idx_train.agregar(h[1])
    final = collections.defaultdict(collections.Counter)
    n = 0
    for alias, rol, copia, filas, h in elegidas:
        if rol != "train" and idx_train.parecido(h[0]):
            final[alias][f"{rol}_parecida_a_train"] += 1
            continue
        nombre = f"{alias}__{n:06d}.jpg"
        n += 1
        destino = salida / "images" / rol / nombre
        os.replace(copia, destino)
        (salida / "labels" / rol / (destino.stem + ".txt")).write_text("\n".join(filas))
        final[alias][f"{rol}_fotos"] += 1
        final[alias][f"{rol}_pistolas"] += sum(int(x.split()[0]) == PISTOLA for x in filas)
        final[alias][f"{rol}_sin_pistola"] += not any(int(x.split()[0]) == PISTOLA for x in filas)
    shutil.rmtree(tmp, ignore_errors=True)
    for alias, c in final.items():
        resumen.setdefault(alias, {}).update({"final": dict(c)})
    tot = collections.Counter()
    for c in final.values():
        tot.update(c)
    resumen["_total"] = dict(tot)
    (salida / "resumen.json").write_text(json.dumps(resumen, indent=1, ensure_ascii=False))
    print("[ext] total: " + json.dumps(dict(tot), ensure_ascii=False), flush=True)
    return resumen


# --------------------------------------------------------------------------- #
def _prueba(tmp: Path) -> int:
    """Tres fuentes sintéticas (YOLO, VOC, video) y una de prueba con un duplicado."""
    import numpy as np
    from PIL import Image
    rng = np.random.default_rng(0)

    def foto(p: Path, semilla: int):
        r = np.random.default_rng(semilla)
        a = (r.random((120, 160, 3)) * 255).astype("uint8")
        Image.fromarray(a).save(p)

    a = tmp / "in" / "a"           # YOLO con data.yaml, clases pistol/phone/rifle
    for s in ("train", "valid"):
        (a / s / "images").mkdir(parents=True)
        (a / s / "labels").mkdir(parents=True)
    (a / "data.yaml").write_text("names: ['Pistol', 'phone', 'rifle', 'person']\n")
    k = 0
    for s in ("train", "valid"):
        for i in range(30):
            foto(a / s / "images" / f"a{k}.jpg", k)
            c = [0, 1, 2, 3][i % 4]
            (a / s / "labels" / f"a{k}.txt").write_text(f"{c} 0.5 0.5 0.2 0.3\n")
            k += 1
    b = tmp / "in" / "b"           # VOC
    (b / "Annotations").mkdir(parents=True)
    (b / "JPEGImages").mkdir(parents=True)
    for i in range(20):
        foto(b / "JPEGImages" / f"b{i}.jpg", 1000 + i)
        (b / "Annotations" / f"b{i}.xml").write_text(
            f"<annotation><size><width>160</width><height>120</height></size><object>"
            f"<name>{'handgun' if i % 2 else 'wallet'}</name><bndbox><xmin>10</xmin><ymin>10</ymin>"
            f"<xmax>60</xmax><ymax>50</ymax></bndbox></object></annotation>")
    c = tmp / "in" / "c"           # cuadros de video + negativos sin etiqueta a propósito
    (c / "images").mkdir(parents=True)
    (c / "labels").mkdir(parents=True)
    (c / "no_weapon").mkdir(parents=True)
    (c / "classes.txt").write_text("gun\n")
    for v in range(6):
        for fr in range(5):
            foto(c / "images" / f"video{v}_frame_{fr:04d}.jpg", 2000 + v * 10 + fr)
            (c / "labels" / f"video{v}_frame_{fr:04d}.txt").write_text("0 0.4 0.4 0.1 0.1\n")
    for i in range(8):
        foto(c / "no_weapon" / f"nw{i}.jpg", 3000 + i)
    d = tmp / "in" / "d"           # prueba: 5 nuevas + 1 copia exacta de una de 'b'
    (d / "images").mkdir(parents=True)
    (d / "labels").mkdir(parents=True)
    (d / "classes.txt").write_text("pistol\n")
    for i in range(5):
        foto(d / "images" / f"d{i}.jpg", 4000 + i)
        (d / "labels" / f"d{i}.txt").write_text("0 0.5 0.5 0.3 0.3\n")
    shutil.copy2(b / "JPEGImages" / "b1.jpg", d / "images" / "copia.jpg")
    (d / "labels" / "copia.txt").write_text("0 0.5 0.5 0.3 0.3\n")
    e = tmp / "in" / "e"           # fuego: 10 llamas, 5 solo "light", 5 llamas mínimas
    (e / "images").mkdir(parents=True)
    (e / "labels").mkdir(parents=True)
    (e / "data.yaml").write_text("names: ['Fire', 'light']\n")
    for i in range(22):            # + 2 llamas chicas (lado 0,012 < lado_min 0,02)
        foto(e / "images" / f"frame_{i:04d}_jpg.rf.{i:032x}.jpg", 5000 + i)
        fila = ("0 0.5 0.5 0.2 0.3" if i < 10 else "1 0.3 0.3 0.1 0.1" if i < 15
                else "0 0.5 0.5 0.001 0.002" if i < 20 else "0 0.5 0.5 0.012 0.012")
        (e / "labels" / f"frame_{i:04d}_jpg.rf.{i:032x}.txt").write_text(fila + "\n")

    cfg = {"fuentes": [
        {"alias": "a", "ruta": "a", "mapa": {"pistol": "pistola", "phone": "celular"},
         "excluir": ["rifle"], "ignorar": ["person"]},
        {"alias": "b", "ruta": "b", "mapa": {"handgun": "pistola"}},     # wallet desconocida
        {"alias": "c", "ruta": "c", "mapa": {"gun": "pistola"}, "vacias": "negativo",
         "video": True},
        {"alias": "d", "ruta": "d", "mapa": {"pistol": "pistola"}, "rol": "prueba"},
        {"alias": "e", "ruta": "e", "mapa": {"fire": "llama"}, "ignorar": ["light"],
         "siempre": ["light"], "tope_neg": 3, "video": True, "bloque": 30, "lado_min": 0.02},
    ]}
    res = armar(cfg, tmp / "salida", None, tmp / "in", frac_val=0.3)
    ok = True
    def chk(cond, msg):
        nonlocal ok
        print(("  ok  " if cond else "  MAL ") + msg)
        ok &= bool(cond)
    chk(res["a"].get("excluye") == 14, "a: 14 fotos con rifle excluidas")
    chk(res["a"]["pos_elegidas"] == 16 and res["a"]["neg_elegidas"] == 30,
        "a: 16 pistolas, 30 negativos (16 celular + 14 persona ignorada->vacía)")
    chk(res["b"].get("desconocida") == 10 and res["b"].get("_clases_desconocidas") == {"wallet": 10},
        "b: wallet no declarada -> 10 fotos afuera con aviso")
    chk(res["c"]["neg_elegidas"] == 8, "c: los 8 no_weapon entran como negativos")
    # split por video: todos los cuadros de un video en el mismo lado
    lados = collections.defaultdict(set)
    for v in range(6):
        for fr in range(5):
            img = tmp / "in" / "c" / "images" / f"video{v}_frame_{fr:04d}.jpg"
            lados[v].add(_al_val("c|" + _grupo_video(img), 0.3))
    chk(all(len(x) == 1 for x in lados.values()), "c: cada video queda entero en train o en val")
    chk(all(p.suffix == ".jpg" and not p.is_symlink() for p in (tmp / "salida" / "images").rglob("*.*")),
        "las fotos son copias locales en JPEG")
    chk(not (tmp / "salida" / "_copias").exists(), "no quedan copias temporales")
    chk(res["d"]["final"].get("prueba_fotos") == 5 and
        (res["d"].get("duplicada", 0) + res["d"]["final"].get("prueba_parecida_a_train", 0)) == 1,
        "d: la copia de una foto de train no entra a prueba")
    neg_lbl = [p for p in (tmp / "salida" / "labels" / "train").glob("a__*")
               if p.read_text().strip().startswith("5 ")]
    chk(len(neg_lbl) > 0, "a: los celulares quedan anotados como clase 5 (celular)")
    chk(res["e"].get("llama_minima") == 5, "e: las 5 llamas mínimas dejan la foto afuera")
    chk(res["e"].get("llama_chica") == 2, "e: las 2 llamas de lado < lado_min también")
    chk(res["e"].get("fijas_sin_tope") == 5 and res["e"]["neg_elegidas"] == 8,
        "e: tope 3 de llamas + las 5 de 'light' (sin tope)")
    e_lbl = sorted((tmp / "salida" / "labels").rglob("e__*.txt"))
    chk(sum(1 for p in e_lbl if p.read_text().strip().startswith("1 ")) == 3
        and sum(1 for p in e_lbl if not p.read_text().strip()) == 5,
        "e: 3 fotos con llama (clase 1) y 5 vacías (la luz no se anota)")
    g = {_grupo_video(Path(f"x/images/frame_{i:04d}_jpg.rf.ab.jpg"), 30) for i in range(60)}
    chk(g == {"images/frame#0", "images/frame#1"}, f"bloque 30: 60 cuadros en 2 grupos ({sorted(g)})")
    chk(_grupo_video(Path("x/images/frame_0391_jpg.rf.ab.jpg")) == "images/frame",
        "sin bloque, el grupo de video de siempre")
    print("PRUEBA", "OK" if ok else "FALLA")
    return 0 if ok else 1


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", default="armas_ext.json")
    ap.add_argument("--salida", default="datasets/armas_ext")
    ap.add_argument("--mezcla", default="datasets/mezcla_v1",
                    help="para sacar lo que se parezca a su val; '' para no usarla")
    ap.add_argument("--base", default="/kaggle/input")
    ap.add_argument("--frac-val", dest="frac_val", type=float, default=0.10)
    ap.add_argument("--prueba", action="store_true")
    ap.add_argument("--revisar", action="store_true",
                    help="solo cuenta y deja hojas de muestras en <salida>/revision/")
    a = ap.parse_args()
    if a.prueba:
        import tempfile
        return _prueba(Path(tempfile.mkdtemp()))
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    mezcla = Path(a.mezcla) if a.mezcla else None
    armar(cfg, Path(a.salida), mezcla if mezcla and mezcla.exists() else None,
          Path(a.base), frac_val=a.frac_val, revisar=a.revisar)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /kaggle/temp/Horus-AI/horus/04_cabezas/objetos/armas_ext.json
{
 "_nota": "06/11 · v4 del YOLO4. Train: Sohas (pistolas + manos con celular/billetera/billete/tarjeta/cuchillo), Monash (pistolas en CCTV) y el handgun de Granada. Prueba (nunca se entrena): Gunmen (CCTV real de ataques; 'Gun' incluye rifles), el sintético de Simuletic y los celulares/billeteras de muzraza.",
 "fuentes": [
  {
   "alias": "sohas",
   "ruta": "datasets/mehmetcubukcu/weapon-detection",
   "formato": "yolo",
   "mapa": {
    "pistol": "pistola",
    "smartphone": "celular",
    "knife": "cuchillo"
   },
   "ignorar": [
    "monedero",
    "billete",
    "tarjeta"
   ],
   "video": true,
   "licencia": "CC BY-SA 4.0 (OD-WeaponDetection, Universidad de Granada)"
  },
  {
   "alias": "monash",
   "ruta": "datasets/muzraza/cctv-gun-detection",
   "formato": "coco",
   "filtro": "^monash_gun/",
   "mapa": {
    "gun": "pistola",
    "guns": "pistola",
    "pistol": "pistola",
    "pistols": "pistola",
    "handgun": "pistola",
    "handguns": "pistola"
   },
   "ignorar": [
    "person",
    "people",
    "human"
   ],
   "video": true,
   "tope_pos": 5000,
   "licencia": "Monash Guns (MIT en el original); el mirror de Kaggle dice 'Other'",
   "_ojo": "Es un mix de Roboflow (CCTV de Monash + fotos de catálogo). Sus fotos sin cajas NO son negativos: la revisión del 06/11 encontró pistolas sin anotar entre ellas."
  },
  {
   "alias": "granada",
   "ruta": "datasets/andrewmvd/handgun-detection",
   "formato": "coco",
   "mapa": {
    "pistol": "pistola",
    "pistols": "pistola",
    "handgun": "pistola",
    "gun": "pistola",
    "guns": "pistola"
   },
   "tope_pos": 1500,
   "licencia": "CC0 (Kaggle, Larxel)"
  },
  {
   "alias": "gunmen",
   "ruta": "datasets/ugorjiir/gun-detection",
   "formato": "yolo",
   "rol": "prueba",
   "mapa": {
    "gun": "pistola"
   },
   "ignorar": [
    "human",
    "person"
   ],
   "licencia": "DbCL (Kaggle)"
  },
  {
   "alias": "sintetico",
   "ruta": "datasets/simuletic/cctv-weapon-dataset",
   "formato": "yolo",
   "rol": "prueba",
   "nombres": [
    "person",
    "gun"
   ],
   "mapa": {
    "gun": "pistola"
   },
   "ignorar": [
    "person"
   ],
   "licencia": "CC BY-SA 4.0 (Simuletic)"
  },
  {
   "alias": "celus",
   "ruta": "datasets/muzraza/cctv-gun-detection",
   "rol": "prueba",
   "filtro": "^(wallet|mobile)/",
   "mapa": {
    "mobile phone": "celular"
   },
   "ignorar": [
    "*"
   ],
   "licencia": "Datacluster Labs / Roboflow (vía muzraza, 'Other')"
  }
 ]
}

In [ ]:
%%writefile /kaggle/temp/Horus-AI/horus/04_cabezas/objetos/fuego_ext.json
{
 "_lado_min": "0,02 = 8 px a 416. Más chico es un punto brillante, igual que un LED. El encendedor del 06/11 midió 0,12 y los fósforos 0,05.",
 "_por_que": "07/11. El 06/11 a la tarde el encendedor dio llama 0,37 y los fósforos 0,38: apenas arriba de 0,35, el umbral con que nace un track. El fuego de mezcla_v1 es casi todo D-Fire (incendios de afuera, grandes). Esto suma llamas chicas de casa: velas, encendedores, fósforos, fuegos de interior y cámaras de seguridad de casas.",
 "fuentes": [
  {"alias": "velas", "ruta": "datasets/sreemantabarman/flame-dataset-candlelightermatch-stick-flames",
   "formato": "yolo", "mapa": {"fire": "llama"}, "video": true, "bloque": 30, "tope_neg": 3500, "lado_min": 0.02,
   "licencia": "CC BY-NC-SA 4.0",
   "_que": "velas, encendedores y fósforos de cerca; cuadros de video aumentados por Roboflow (rotados, con bordes negros). Las copias de un mismo cuadro van juntas al split (bloque de 30 cuadros)."},
  {"alias": "hogar", "ruta": "datasets/pengbo00/home-fire-dataset",
   "formato": "yolo", "nombres": ["fire", "smoke"], "mapa": {"fire": "llama", "smoke": "humo"},
   "vacias": "negativo", "lado_min": 0.02,
   "licencia": "CC BY-NC 4.0",
   "_que": "fuegos chicos dentro de casas, muchas cámaras de seguridad. Sin data.yaml: 0=fire, 1=smoke (visto en la hoja del relevamiento). Las 135 fotos sin cajas son casas sin fuego (livings, oficinas, lámparas): negativos a propósito."},
  {"alias": "iw", "ruta": "datasets/ironwolf437/fire-detection-dataset",
   "formato": "yolo", "mapa": {"fire": "llama", "smoke": "humo"},
   "ignorar": ["light", "nonfire"], "siempre": ["light", "nonfire"], "video": true, "tope_neg": 4000,
   "vacias": "negativo", "lado_min": 0.02,
   "licencia": "Apache 2.0",
   "_que": "mezcla de CCTV (oficinas, negocios, casas), fuegos de prueba y AI-Hub. light/nonfire son negativos difíciles anotados a propósito (luces, cosas que parecen fuego): la caja se tira y la foto entra entera, sin tope. Las 497 sin cajas son cuadros de CCTV sin fuego (negocios, cajeros): negativos."},
  {"alias": "interior", "ruta": "datasets/sinchanashivanand/indoor-fire-and-smoke-detection-with-yolov8",
   "formato": "yolo", "rol": "prueba", "mapa": {"fire": "llama", "smoke": "humo"}, "tope_neg": 1500,
   "licencia": "CC0",
   "_que": "fuegos de interior a 256 px. PRUEBA: el modelo no la ve. Ojo: comparte videos de origen con iw (part*_mp4); lo que se parezca a train se saca por dHash, pero las copias aumentadas pueden pasar."}
 ]
}

In [ ]:
%cd {OBJ}
!python armas_ext.py --config armas_ext.json --salida datasets/armas_ext \
    --mezcla datasets/mezcla_v1 --base /kaggle/input
!python armas_ext.py --config fuego_ext.json --salida datasets/fuego_ext \
    --mezcla datasets/mezcla_v1 --base /kaggle/input
print(open("datasets/armas_ext/resumen.json").read()[:3000])
print(open("datasets/fuego_ext/resumen.json").read()[:6000])

## 5 · El script de entrenamiento (el mismo que queda en el repo)

In [ ]:
%%writefile /kaggle/temp/Horus-AI/horus/04_cabezas/objetos/entrenar_yolo.py
# -*- coding: utf-8 -*-
"""
entrenar_yolo.py · YOLO11 para lo que COCO no sabe: pistola, humo, llama, paquete.

26/09. El detector preentrenado (motor_yolo.py) ya se encarga de persona,
cuchillo y celular. Para las otras cuatro clases seguía corriendo la cabeza
propia arriba de un ResNet-50, que es lo más lento de todo el servicio. Este
script entrena un YOLO11 chico SOLO con esas cuatro, partiendo de los pesos de
COCO (que ya saben ver bordes, texturas, objetos), con el mismo dataset que
armaba la cabeza propia.

Por qué cuatro clases y no siete
--------------------------------
El dataset mezcla fuentes que no anotan lo mismo: D-Fire y Pyro-SDIS tienen
personas SIN etiquetar. La cabeza propia lo resolvía ignorando la clase persona
en esas fotos (v4); YOLO no tiene esa opción, y entrenar persona así le
enseñaría que "la gente en una foto de incendio es fondo". Con cuatro clases el
problema desaparece: para este modelo una persona es fondo SIEMPRE, y la
persona la sigue viendo el YOLO de COCO, que fue entrenado con cientos de miles.

    python entrenar_yolo.py --mezcla datasets/mezcla_v1 --propio auto
    python entrenar_yolo.py --mezcla datasets/mezcla_v1 --propio auto \
        --extra datasets/armas_ext --comparar ruta/a/horus_yolo4_v3.pt
    python entrenar_yolo.py --prueba          # un minuto en CPU, para ver que anda

--extra (06/11): una carpeta de armas_ext.py con datasets públicos de armas.
Su train entra ENTERO (los negativos de ahí son los difíciles: manos con
celular, billetera, tarjeta, cuchillo); su val y su prueba se miden aparte, así
el val de mezcla_v1 sigue siendo comparable con las versiones anteriores.

07/11 (v5): --extra acepta varias carpetas (armas_ext y fuego_ext, la de llamas
chicas de casa). Cada una aporta su val_<nombre> y su prueba_<nombre>. La medida
por foto ahora es para las cuatro clases en una sola pasada: lo que más importa
de la llama es en cuántas fotos SIN fuego nace un track (una llama sola da
ALERTA a los 1,5 s). Con --modelo <pesos.pt de la v4> se afina la v4 en vez de
arrancar de COCO: la mitad de épocas para el mismo resultado.

    python entrenar_yolo.py --mezcla datasets/mezcla_v1 --propio auto \
        --extra datasets/armas_ext datasets/fuego_ext \
        --modelo v4/horus_yolo4.pt --comparar v4/horus_yolo4.pt --epocas 40 \
        --lr0 0.003 --optimizador SGD --warmup 1 --paciencia 0 --val-con-extras

Lección de la primera v5 (07/11, run 366713820): afinando con lo de siempre
(optimizer auto = MuSGD a 0,01, que además IGNORA --lr0, y warmup de 3 épocas
con bias a 0,1) el val se cayó de 0,67 a 0,51 en la época 3 y recién volvía a
0,67 en la 21. La paciencia de 20 cortó ahí y best.pt quedó en la época 1:
casi la v4. Por eso: SGD a 0,003, una época de warmup sin el bias alto, sin
corte temprano (--paciencia 0), y best.pt se elige con el val de las tres
fuentes juntas (--val-con-extras), no solo con el de mezcla_v1 (que no tiene
una sola vela). val_mezcla se sigue midiendo aparte, igual que siempre.

Sale `horus_yolo4.pt` y `horus_yolo4_<tam>.onnx`. Van a `modelos/` y el servicio
los usa solos (ver `motor_yolo.pesos_horus_por_defecto`).
"""

from __future__ import annotations

import argparse
import json
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

CLASES_7 = ("humo", "llama", "persona", "pistola", "cuchillo", "celular", "paquete")
CLASES_YOLO = ("pistola", "humo", "llama", "paquete")
REMAP = {CLASES_7.index(c): i for i, c in enumerate(CLASES_YOLO)}
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
def _carpetas(mezcla: Path, split: str) -> Tuple[Path, Path]:
    """Las dos disposiciones que conviven: images/<split> y <split>/images."""
    for img, lbl in ((mezcla / "images" / split, mezcla / "labels" / split),
                     (mezcla / split / "images", mezcla / split / "labels")):
        if img.is_dir():
            return img, lbl
    raise SystemExit(f"no encuentro las imágenes de '{split}' en {mezcla}")


def _enlazar(origen: Path, destino: Path) -> None:
    """Symlink si se puede (Kaggle, Linux); copia si no (Windows sin permisos)."""
    if destino.exists() or destino.is_symlink():
        destino.unlink()
    try:
        destino.symlink_to(origen.resolve())
    except OSError:
        shutil.copy2(origen, destino)


def _propio(raiz: Optional[Path]) -> List[Tuple[Path, str]]:
    """Frames de la cámara propia que son negativos SEGUROS para las 4 clases.

    Solo las escenas que declaran ausentes a las cuatro (la "vacia" las declara
    a todas). Una escena donde alguna podría estar no se usa: YOLO no tiene
    forma de decir "de esto no opino" y una caja faltante le enseña que el
    objeto es fondo.
    """
    if raiz is None:
        return []
    man = raiz / "horus_propio.json"
    if not man.exists():
        hallados = sorted(raiz.rglob("horus_propio.json"))
        if not hallados:
            print(f"[propio] no hay horus_propio.json en {raiz}")
            return []
        man = hallados[0]
    datos = json.loads(man.read_text(encoding="utf-8"))
    out = []
    for escena, info in (datos.get("escenas") or {}).items():
        ausentes = set(info.get("ausentes") or [])
        # 26/09: la vacia se revisó a mano (ninguna de las 7 clases); los
        # manifiestos viejos solo declaraban persona/pistola/cuchillo.
        if escena == "vacia":
            ausentes = set(CLASES_7)
        if not set(CLASES_YOLO) <= ausentes:
            print(f"[propio] escena '{escena}': no declara ausentes a las 4 clases, la salteo")
            continue
        fotos = sorted(p for p in (man.parent / escena / "images").glob("*")
                       if p.suffix.lower() in EXT)
        out += [(f, escena) for f in fotos]
        print(f"[propio] escena '{escena}': {len(fotos)} frames como negativos")
    return out


def _extra(extra: Optional[Path], parte: str) -> List[Tuple[Path, List[str]]]:
    """(foto, filas de 4 clases) de una parte de la carpeta de armas_ext.py."""
    if extra is None:
        return []
    dimg, dlbl = extra / "images" / parte, extra / "labels" / parte
    if not dimg.is_dir():
        return []
    out = []
    for foto in sorted(p for p in dimg.iterdir() if p.suffix.lower() in EXT):
        lbl = dlbl / (foto.stem + ".txt")
        filas = []
        if lbl.exists():
            for linea in lbl.read_text().splitlines():
                partes = linea.split()
                if len(partes) == 5 and int(partes[0]) in REMAP:
                    filas.append(f"{REMAP[int(partes[0])]} " + " ".join(partes[1:]))
        out.append((foto, filas))
    return out


def ruta_val_mezcla(destino: Path) -> Path:
    """data.yaml del val PURO de mezcla_v1 cuando el de entrenamiento suma los extras."""
    return destino.parent / f"{destino.name}_val_mezcla" / "data.yaml"


def nombre_extra(extra: Path) -> str:
    """datasets/armas_ext -> 'armas'; datasets/fuego_ext -> 'fuego'."""
    n = Path(extra).name
    return n[:-4] if n.endswith("_ext") else n


def armar_eval(extra: Optional[Path], destino: Path, parte: str,
               etiqueta: Optional[str] = None) -> Optional[Path]:
    """Un dataset solo-val con la parte `parte` del extra; devuelve su data.yaml."""
    fotos = _extra(extra, parte)
    if not fotos:
        return None
    etiqueta = etiqueta or nombre_extra(extra)
    d = destino.parent / f"{destino.name}_{parte}_{etiqueta}"
    if d.exists():
        shutil.rmtree(d)
    (d / "images" / "val").mkdir(parents=True)
    (d / "labels" / "val").mkdir(parents=True)
    for foto, filas in fotos:
        _enlazar(foto, d / "images" / "val" / foto.name)
        (d / "labels" / "val" / (foto.stem + ".txt")).write_text("\n".join(filas))
    yaml = d / "data.yaml"
    yaml.write_text(
        f"path: {d.resolve()}\ntrain: images/val\nval: images/val\n"
        "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASES_YOLO)),
        encoding="utf-8")
    con = sum(1 for _, f in fotos if f)
    print(f"[dataset] {parte} de {etiqueta}: {len(fotos)} fotos ({con} con caja, "
          f"{len(fotos) - con} sin ninguna)")
    return yaml


def armar(mezcla: Path, destino: Path, propio: Optional[Path] = None,
          repetir_propio: int = 4, frac_negativos: float = 0.25,
          limite: Optional[int] = None, semilla: int = 0,
          extra: Union[None, Path, Sequence[Path]] = None,
          val_extras: bool = False) -> Path:
    """Arma el dataset de 4 clases y devuelve la ruta al data.yaml.

    val_extras: el val del entrenamiento (con el que se elige best.pt) suma el
    val de cada extra; el val puro de mezcla_v1 queda en ruta_val_mezcla().
    """
    rnd = random.Random(semilla)
    extras = [extra] if isinstance(extra, (str, Path)) else list(extra or [])
    if destino.exists():
        shutil.rmtree(destino)
    puro = ruta_val_mezcla(destino).parent
    if puro.exists():
        shutil.rmtree(puro)
    cuenta: Dict[str, Counter] = {}
    for split in ("train", "val"):
        dimg, dlbl = _carpetas(mezcla, split)
        oimg, olbl = destino / "images" / split, destino / "labels" / split
        oimg.mkdir(parents=True, exist_ok=True)
        olbl.mkdir(parents=True, exist_ok=True)
        positivas, negativas = [], []
        fotos = sorted(p for p in dimg.iterdir() if p.suffix.lower() in EXT)
        if limite:
            fotos = fotos[:limite]
        for foto in fotos:
            lbl = dlbl / (foto.stem + ".txt")
            filas = []
            if lbl.exists():
                for linea in lbl.read_text().splitlines():
                    partes = linea.split()
                    if len(partes) == 5 and int(partes[0]) in REMAP:
                        filas.append(f"{REMAP[int(partes[0])]} " + " ".join(partes[1:]))
            (positivas if filas else negativas).append((foto, filas))
        # Fondo: hace falta (es lo que enseña a NO disparar), pero si son la
        # mitad del dataset el modelo aprende que casi nunca hay nada.
        max_neg = int(len(positivas) * frac_negativos / max(1e-6, 1 - frac_negativos))
        rnd.shuffle(negativas)
        elegidas = positivas + negativas[:max_neg]
        c = Counter()
        for foto, filas in elegidas:
            _enlazar(foto, oimg / foto.name)
            (olbl / (foto.stem + ".txt")).write_text("\n".join(filas))
            for f in filas:
                c[CLASES_YOLO[int(f.split()[0])]] += 1
            c["_fotos"] += 1
            c["_fondo"] += not filas
        if split == "train":
            for foto, escena in _propio(propio):
                for k in range(max(1, repetir_propio)):
                    nombre = f"propio-{escena}__{foto.stem}_r{k}{foto.suffix.lower()}"
                    _enlazar(foto, oimg / nombre)
                    (olbl / (Path(nombre).stem + ".txt")).write_text("")
                    c["_fotos"] += 1
                    c["_propio"] += 1
            # 06/11: datasets públicos de armas. Entran enteros, sin el tope de
            # negativos: los de ahí son "mano con algo que no es un arma", que
            # es justo lo que le faltaba a la v3 (la media del 06/11).
            # 07/11: lo mismo con fuego_ext (llamas chicas de casa y sus
            # negativos difíciles: lámparas, luces).
            for ex in extras:
                ne = nombre_extra(ex)
                for foto, filas in _extra(ex, "train"):
                    _enlazar(foto, oimg / foto.name)
                    (olbl / (foto.stem + ".txt")).write_text("\n".join(filas))
                    for f in filas:
                        c[CLASES_YOLO[int(f.split()[0])]] += 1
                    c["_fotos"] += 1
                    c["_fondo"] += not filas
                    c["_extra"] += 1
                    c[f"_extra_{ne}"] += 1
        if split == "val" and val_extras and extras:
            # primero, una copia del val puro de mezcla_v1 (la vara de siempre)
            (puro / "images" / "val").mkdir(parents=True, exist_ok=True)
            (puro / "labels" / "val").mkdir(parents=True, exist_ok=True)
            for foto in sorted(oimg.iterdir()):
                _enlazar(foto, puro / "images" / "val" / foto.name)
                shutil.copy2(olbl / (foto.stem + ".txt"), puro / "labels" / "val" / (foto.stem + ".txt"))
            (puro / "data.yaml").write_text(
                f"path: {puro.resolve()}\ntrain: images/val\nval: images/val\n"
                "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASES_YOLO)),
                encoding="utf-8")
            for ex in extras:
                ne = nombre_extra(ex)
                for foto, filas in _extra(ex, "val"):
                    _enlazar(foto, oimg / foto.name)
                    (olbl / (foto.stem + ".txt")).write_text("\n".join(filas))
                    for f in filas:
                        c[CLASES_YOLO[int(f.split()[0])]] += 1
                    c["_fotos"] += 1
                    c["_fondo"] += not filas
                    c["_extra"] += 1
                    c[f"_extra_{ne}"] += 1
        cuenta[split] = c

    yaml = destino / "data.yaml"
    yaml.write_text(
        f"path: {destino.resolve()}\ntrain: images/train\nval: images/val\n"
        "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASES_YOLO)),
        encoding="utf-8")
    for split, c in cuenta.items():
        de_extras = ", ".join(f"{k[7:]} {v}" for k, v in sorted(c.items()) if k.startswith("_extra_"))
        print(f"[dataset] {split}: {c['_fotos']} fotos ({c['_fondo']} de fondo, "
              f"{c['_propio']} de tu cámara, {c['_extra']} de extras: {de_extras or '-'}) · cajas: "
              + ", ".join(f"{n}={c[n]}" for n in CLASES_YOLO))
    vacias = [n for n in CLASES_YOLO if cuenta["train"][n] == 0]
    if vacias:
        raise SystemExit(f"sin una sola caja de {vacias} en train: no entreno a ciegas")
    return yaml


# --------------------------------------------------------------------------- #
# Entrenar y exportar
# --------------------------------------------------------------------------- #
def entrenar(data_yaml: Path, modelo: str = "yolo11n.pt", tam: int = 416,
             epocas: int = 80, batch: int = -1, device: str = "",
             paciencia: int = 20, proyecto: str = "runs_horus",
             nombre: str = "yolo4", workers: int = 4,
             lr0: Optional[float] = None, optimizador: Optional[str] = None,
             warmup: Optional[float] = None) -> Path:
    from ultralytics import YOLO
    m = YOLO(modelo)
    extra_args: dict = {}
    if lr0:
        # con optimizer="auto" ultralytics ignora lr0 (pasó el 07/11): hay que nombrarlo
        extra_args.update(lr0=lr0, optimizer=optimizador or "SGD")
    elif optimizador:
        extra_args["optimizer"] = optimizador
    if warmup is not None:
        # sin el bias a 0,1 del warmup de siempre: a un modelo ya entrenado lo desarma
        extra_args.update(warmup_epochs=warmup, warmup_bias_lr=lr0 or 0.01)
    r = m.train(
        data=str(data_yaml), imgsz=tam, epochs=epocas, batch=batch,
        device=device or None, patience=paciencia, project=proyecto, name=nombre,
        exist_ok=True, seed=0, workers=workers, cos_lr=True, close_mosaic=10,
        # La realidad es peor que las fotos del dataset: luz de más o de menos,
        # colores corridos, cuadros movidos. Aumentación fuerte de brillo y
        # algo de rotación; el desenfoque lo agrega Albumentations si está.
        hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, degrees=5.0, mixup=0.1,
        plots=False, verbose=True, **extra_args)
    # 05/11: el ultralytics de ahora guarda un `project` relativo adentro de
    # runs/<tarea>/ (quedó en runs/detect/runs_horus/yolo4) y acá se buscaba
    # runs_horus/yolo4: 1,4 h de T4 entrenadas y nada exportado. La carpeta
    # real la sabe el entrenador; las otras quedan como respaldo.
    candidatas = []
    for base in (getattr(getattr(m, "trainer", None), "save_dir", None),
                 getattr(r, "save_dir", None)):
        if base:
            candidatas.append(Path(base) / "weights" / "best.pt")
    candidatas += [Path(proyecto) / nombre / "weights" / "best.pt",
                   Path("runs") / "detect" / proyecto / nombre / "weights" / "best.pt"]
    best = next((p for p in candidatas if p.exists()), None)
    if best is None:
        raise SystemExit("no se generó best.pt; lo busqué en: "
                         + ", ".join(str(p) for p in candidatas))
    print(f"[entrenar] pesos: {best}")
    return best


def exportar(best: Path, salida: Path, tam: int = 416) -> Dict[str, Path]:
    from ultralytics import YOLO
    salida.mkdir(parents=True, exist_ok=True)
    pt = salida / "horus_yolo4.pt"
    shutil.copy2(best, pt)
    onnx = Path(YOLO(str(best)).export(format="onnx", imgsz=tam, simplify=True))
    destino_onnx = salida / f"horus_yolo4_{tam}.onnx"
    shutil.move(str(onnx), destino_onnx)
    print(f"[exportar] {pt.name} ({pt.stat().st_size / 1e6:.1f} MB) · "
          f"{destino_onnx.name} ({destino_onnx.stat().st_size / 1e6:.1f} MB)")
    return {"pt": pt, "onnx": destino_onnx}


def guardar_ultimo(best: Path, salida: Path) -> Optional[Path]:
    """Copia last.pt (la última época) junto a horus_yolo4.pt, para medirlo también:
    best.pt es la mejor época según el val del entrenamiento, que no es todo."""
    last = Path(best).parent / "last.pt"
    if not last.exists():
        return None
    salida.mkdir(parents=True, exist_ok=True)
    dst = salida / "horus_yolo4_ultimo.pt"
    shutil.copy2(last, dst)
    try:
        from ultralytics.utils.torch_utils import strip_optimizer
        strip_optimizer(str(dst))
    except Exception as e:
        print(f"[ultimo] quedó con el optimizador adentro: {e!r}")
    print(f"[ultimo] {dst.name} ({dst.stat().st_size / 1e6:.1f} MB)")
    return dst


def validar(pt: Path, data_yaml: Path, tam: int = 416, nombre: str = "val") -> Dict[str, float]:
    from ultralytics import YOLO
    r = YOLO(str(pt)).val(data=str(data_yaml), imgsz=tam, verbose=False, plots=False,
                          project=str(Path(data_yaml).parent / "_val"), name=Path(pt).stem,
                          exist_ok=True)
    aps = {CLASES_YOLO[int(i)]: float(r.box.ap50[k]) for k, i in enumerate(r.box.ap_class_index)}
    print(f"[{nombre}] {Path(pt).name}: mAP50={r.box.map50:.3f} · "
          + " ".join(f"{n}={aps[n]:.2f}" for n in CLASES_YOLO if n in aps), flush=True)
    return aps


def por_imagen(pt: Path, data_yaml: Path, tam: int = 416,
               clases: Sequence[str] = CLASES_YOLO,
               umbrales=(0.35, 0.50), nombre: str = "val") -> Dict[str, dict]:
    """Lo que ve el servicio: ¿nace un track de cada clase en la foto?

    El AP no dice cuántas fotos SIN arma (o sin fuego) disparan al umbral con
    el que nace un track (0,35). Esto sí: por foto, la confianza máxima de cada
    clase, contra si la foto tiene o no una caja de esa clase. Una sola pasada
    del modelo para todas las clases. Devuelve {clase: {...}}.
    """
    from ultralytics import YOLO
    m = YOLO(str(pt))
    d = Path(data_yaml).parent
    fotos = sorted(p for p in (d / "images" / "val").iterdir() if p.suffix.lower() in EXT)
    filas = []             # (fuente, {clase: conf máx}, {clases anotadas})
    for i in range(0, len(fotos), 64):
        lote = fotos[i:i + 64]
        for foto, r in zip(lote, m.predict([str(p) for p in lote], imgsz=tam, conf=0.05,
                                           verbose=False, stream=False)):
            cl = r.boxes.cls.tolist() if r.boxes is not None else []
            cf = r.boxes.conf.tolist() if r.boxes is not None else []
            mx = {c: max([s for k, s in zip(cl, cf) if int(k) == CLASES_YOLO.index(c)], default=0.0)
                  for c in clases}
            lbl = d / "labels" / "val" / (foto.stem + ".txt")
            anotadas = set()
            if lbl.exists():
                for l in lbl.read_text().splitlines():
                    t = l.split()
                    if t and t[0].isdigit() and int(t[0]) < len(CLASES_YOLO):
                        anotadas.add(CLASES_YOLO[int(t[0])])
            fuente = foto.name.split("__")[0] if "__" in foto.name else "todo"
            filas.append((fuente, mx, anotadas))

    def resumir(c, s_):
        o = {"fotos_con": len(c), "fotos_sin": len(s_)}
        for u in umbrales:
            o[f"recall_{u:.2f}"] = round(sum(x >= u for x in c) / max(1, len(c)), 4)
            o[f"falsas_{u:.2f}"] = round(sum(x >= u for x in s_) / max(1, len(s_)), 4)
        return o

    def linea(o):
        ve = lambda u: f"ve {o[f'recall_{u:.2f}']:.1%} / " if o["fotos_con"] else ""
        return (f"{o['fotos_con']} con, {o['fotos_sin']} sin · "
                + " · ".join(f"@{u:.2f} {ve(u)}dispara en "
                             f"{o[f'falsas_{u:.2f}']:.1%} de las sin" for u in umbrales))

    res: Dict[str, dict] = {}
    fuentes = sorted({f for f, _, _ in filas})
    for clase in clases:
        con = [(f, mx[clase]) for f, mx, an in filas if clase in an]
        sin = [(f, mx[clase]) for f, mx, an in filas if clase not in an]
        out = resumir([x for _, x in con], [x for _, x in sin])
        print(f"[{nombre}] {Path(pt).name} por foto ({clase}): " + linea(out), flush=True)
        if len(fuentes) > 1:
            out["por_fuente"] = {}
            for fu in fuentes:
                o = resumir([x for f, x in con if f == fu], [x for f, x in sin if f == fu])
                out["por_fuente"][fu] = o
                print(f"    {fu:<12} " + linea(o), flush=True)
        res[clase] = out
    return res


def medir_todo(pesos: Dict[str, Path], yamls: Dict[str, Path], tam: int,
               salida: Path) -> dict:
    """AP50 y la medida por foto (las 4 clases), para cada modelo en cada dataset;
    a metricas.json: {modelo: {dataset: {"ap50": {...}, "por_foto": {clase: {...}}}}}."""
    res: Dict[str, dict] = {}
    for nm, pt in pesos.items():
        if pt is None or not Path(pt).exists():
            continue
        for ny, y in yamls.items():
            if y is None:
                continue
            try:
                res.setdefault(nm, {})[ny] = {"ap50": validar(pt, y, tam, ny),
                                              "por_foto": por_imagen(pt, y, tam, nombre=ny)}
            except Exception as e:     # una medida rota no puede tirar el entrenamiento
                print(f"[medir] {nm} en {ny}: {e!r}", flush=True)
    salida.mkdir(parents=True, exist_ok=True)
    (salida / "metricas.json").write_text(json.dumps(res, indent=1, ensure_ascii=False))
    return res


# --------------------------------------------------------------------------- #
def _prueba_sintetica(dest: Path) -> Path:
    """Un mini dataset de mentira en el formato de mezcla_v1: 7 clases."""
    import numpy as np
    import cv2
    rnd = np.random.default_rng(0)
    for split, n in (("train", 24), ("val", 8)):
        (dest / "images" / split).mkdir(parents=True, exist_ok=True)
        (dest / "labels" / split).mkdir(parents=True, exist_ok=True)
        for i in range(n):
            img = np.full((240, 320, 3), 90, np.uint8)
            filas = []
            for c in (i % 7, (i + 3) % 7):
                x, y = rnd.integers(20, 250), rnd.integers(20, 170)
                cv2.rectangle(img, (int(x), int(y)), (int(x) + 50, int(y) + 50),
                              tuple(int(v) for v in rnd.integers(0, 255, 3)), -1)
                filas.append(f"{c} {(x + 25) / 320:.4f} {(y + 25) / 240:.4f} "
                             f"{50 / 320:.4f} {50 / 240:.4f}")
            cv2.imwrite(str(dest / "images" / split / f"f{i}.jpg"), img)
            (dest / "labels" / split / f"f{i}.txt").write_text("\n".join(filas))
    return dest


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--mezcla", default="datasets/mezcla_v1")
    ap.add_argument("--propio", default="auto",
                    help='carpeta de grabar_propio.py, "auto" (la busca en /kaggle/input '
                         'y en datasets/propio) o "no"')
    ap.add_argument("--destino", default="datasets/yolo4")
    ap.add_argument("--modelo", default="yolo11n.pt")
    ap.add_argument("--tam", type=int, default=416)
    ap.add_argument("--epocas", type=int, default=80)
    ap.add_argument("--batch", type=int, default=-1)
    ap.add_argument("--device", default="")
    ap.add_argument("--salida", default="modelos_yolo")
    ap.add_argument("--proyecto", default="runs_horus",
                    help="carpeta de las corridas; en Kaggle conviene una de "
                         "/kaggle/working para que los pesos queden en Output")
    ap.add_argument("--extra", nargs="*", default=[],
                    help="carpetas de armas_ext.py (armas_ext, fuego_ext, ...)")
    ap.add_argument("--comparar", default="",
                    help="pesos .pt de la versión anterior, para medirla igual")
    ap.add_argument("--lr0", type=float, default=None,
                    help="tasa inicial (con SGD salvo --optimizador); sin esto, la de ultralytics")
    ap.add_argument("--optimizador", default=None, help="SGD, AdamW, MuSGD...; sin esto, auto")
    ap.add_argument("--warmup", type=float, default=None,
                    help="épocas de warmup, sin el bias alto (para afinar un modelo ya entrenado)")
    ap.add_argument("--paciencia", type=int, default=20,
                    help="épocas sin mejorar antes de cortar; 0 = no cortar nunca")
    ap.add_argument("--val-con-extras", dest="val_con_extras", action="store_true",
                    help="best.pt se elige con el val de mezcla_v1 + el de cada extra")
    ap.add_argument("--solo-armar", dest="solo_armar", action="store_true")
    ap.add_argument("--prueba", action="store_true",
                    help="dataset sintético chico y 1 época: verifica el camino entero")
    a = ap.parse_args()

    if a.prueba:
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        mezcla = _prueba_sintetica(tmp / "mezcla")
        extras = []
        for nom, pos in (("armas_ext", "3"), ("fuego_ext", "1")):      # pistola / llama (7 clases)
            ext = tmp / nom
            extras.append(ext)
            for parte in ("train", "val", "prueba"):
                (ext / "images" / parte).mkdir(parents=True)
                (ext / "labels" / parte).mkdir(parents=True)
                for f in sorted((mezcla / "images" / ("val" if parte != "train" else "train")).iterdir())[:6]:
                    shutil.copy2(f, ext / "images" / parte / f"{nom[:2]}__{parte}_{f.name}")
                    (ext / "labels" / parte / f"{nom[:2]}__{parte}_{f.stem}.txt").write_text(
                        f"{pos} 0.5 0.5 0.2 0.2" if f.stem.endswith(("0", "2", "4")) else "5 0.5 0.5 0.2 0.2")
        yaml = armar(mezcla, tmp / "yolo4", None, extra=extras, val_extras=True)
        yamls = {"val_mezcla": ruta_val_mezcla(tmp / "yolo4")}
        n_val = len(list((tmp / "yolo4" / "images" / "val").iterdir()))
        n_puro = len(list((ruta_val_mezcla(tmp / "yolo4").parent / "images" / "val").iterdir()))
        assert n_puro > 0 and n_val == n_puro + 12, (n_val, n_puro)    # 6 de val por extra
        for ext in extras:
            yamls[f"val_{nombre_extra(ext)}"] = armar_eval(ext, tmp / "yolo4", "val")
            yamls[f"prueba_{nombre_extra(ext)}"] = armar_eval(ext, tmp / "yolo4", "prueba")
        best = entrenar(yaml, a.modelo, tam=160, epocas=1, batch=8, device="cpu",
                        proyecto=str(tmp / "runs"), workers=0)
        rutas = exportar(best, tmp / "salida", tam=160)
        # afinar desde los pesos recién hechos (como la v5 desde la v4)
        best2 = entrenar(yaml, str(rutas["pt"]), tam=160, epocas=1, batch=8, device="cpu",
                         proyecto=str(tmp / "runs2"), workers=0, lr0=0.005, optimizador="SGD",
                         warmup=1, paciencia=0)
        assert best2.exists(), best2
        ultimo = guardar_ultimo(best2, tmp / "salida")
        assert ultimo is not None and ultimo.exists(), ultimo
        res = medir_todo({"nuevo": rutas["pt"], "ultimo": ultimo, "anterior": rutas["pt"]},
                         yamls, 160, tmp / "salida")
        esperados = {"val_mezcla", "val_armas", "prueba_armas", "val_fuego", "prueba_fuego"}
        assert set(res.get("nuevo", {})) == esperados, res.get("nuevo", {}).keys()
        assert set(res.get("ultimo", {})) == esperados, res.get("ultimo", {}).keys()
        pf = res["nuevo"]["val_fuego"]["por_foto"]
        assert set(pf) == set(CLASES_YOLO) and pf["llama"]["fotos_con"] == 3, pf["llama"]
        assert res["nuevo"]["val_armas"]["por_foto"]["pistola"]["fotos_con"] == 3
        from motor_yolo import ConfigYolo, MotorYolo
        m = MotorYolo(ConfigYolo(pesos=str(rutas["onnx"]), tam=160, verboso=True))
        ok = set(m.mapa.values()) == set(CLASES_YOLO)
        print("PRUEBA", "OK" if ok else "FALLA", "· clases que ve el motor:", sorted(m.mapa.values()))
        return 0 if ok else 1

    propio: Optional[Path] = None
    if a.propio == "auto":
        for base in (Path("/kaggle/input"), Path("datasets/propio")):
            if base.exists():
                hall = sorted(base.rglob("horus_propio.json"))
                if hall:
                    propio = hall[0].parent
                    break
    elif a.propio != "no":
        propio = Path(a.propio)
    extras = [Path(e) for e in a.extra if e]
    for ex in extras:
        if not (ex / "images" / "train").is_dir():
            raise SystemExit(f"--extra {ex}: no tiene images/train (¿corrió armas_ext.py?)")
    yaml = armar(Path(a.mezcla), Path(a.destino), propio, extra=extras,
                 val_extras=a.val_con_extras)
    yamls = {"val_mezcla": ruta_val_mezcla(Path(a.destino)) if a.val_con_extras and extras else yaml}
    for ex in extras:
        yamls[f"val_{nombre_extra(ex)}"] = armar_eval(ex, Path(a.destino), "val")
        yamls[f"prueba_{nombre_extra(ex)}"] = armar_eval(ex, Path(a.destino), "prueba")
    if a.solo_armar:
        return 0
    best = entrenar(yaml, a.modelo, tam=a.tam, epocas=a.epocas, batch=a.batch,
                    device=a.device, proyecto=a.proyecto, lr0=a.lr0, paciencia=a.paciencia,
                    optimizador=a.optimizador, warmup=a.warmup)
    rutas = exportar(best, Path(a.salida), tam=a.tam)
    pesos = {"nuevo": rutas["pt"]}
    ultimo = guardar_ultimo(best, Path(a.salida))
    if ultimo:
        pesos["ultimo"] = ultimo
    pesos["anterior"] = Path(a.comparar) if a.comparar else None
    medir_todo(pesos, yamls, a.tam, Path(a.salida))
    return 0


if __name__ == "__main__":
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    raise SystemExit(main())

## 6 · Armar el dataset de 4 clases, afinar la v4, exportar y medir

Se corre como proceso aparte (`!python`) y no en el kernel: el entrenamiento con dos
GPUs lanza procesos hijos (DDP) y desde el kernel de Jupyter no puede.

07/11: se parte de los pesos de la v4 (`--modelo`) y no de COCO: ya sabe pistola, humo,
llama y paquete, y con 40 épocas aprende lo nuevo sin olvidar lo viejo (lo viejo sigue
en el dataset). La v4 se mide con lo mismo (`--comparar`).

v5b: `--lr0 0.003 --optimizador SGD --warmup 1 --paciencia 0 --val-con-extras`. Ver el
encabezado: la primera v5 se desarmó con la tasa de siempre y se cortó en la época 21.

In [ ]:
%cd {OBJ}
DEV = ",".join(str(i) for i in range(NGPU))
# Con dos GPUs ultralytics no acepta batch automático (-1): 32 por placa.
# --proyecto en /kaggle/working: si algo falla después de entrenar, best.pt
# queda igual en Output (el 05/11 se perdió entero en /kaggle/temp).
BATCH = 32 * max(1, NGPU)
if ANTERIOR:
    MODELO, EPOCAS, COMPARAR = str(ANTERIOR), 40, f"--comparar {ANTERIOR}"
    # afinar: tasa baja con SGD, warmup corto sin el bias alto, sin corte temprano
    AFINAR = "--lr0 0.003 --optimizador SGD --warmup 1 --paciencia 0"
else:
    MODELO, EPOCAS, COMPARAR, AFINAR = "yolo11n.pt", 80, "", "--paciencia 0"
print("parte de:", MODELO, "·", EPOCAS, "épocas")
!python entrenar_yolo.py --mezcla datasets/mezcla_v1 --propio {PROPIO} \
    --extra datasets/armas_ext datasets/fuego_ext --modelo {MODELO} {COMPARAR} \
    {AFINAR} --val-con-extras \
    --destino /kaggle/temp/yolo4 --tam 416 --epocas {EPOCAS} --batch {BATCH} --device {DEV} \
    --salida {TRABAJO}/modelos_yolo --proyecto {TRABAJO}/runs_yolo

## 7 · Bajarlo

In [ ]:
import base64, hashlib, json, lzma
for f in sorted(pathlib.Path(f"{TRABAJO}/modelos_yolo").glob("*")):
    print(f"  {f.name:<26} {f.stat().st_size/1e6:>6.1f} MB")
print("\nEn Output -> modelos_yolo/. Los dos van a horus/04_cabezas/objetos/modelos/")
print("en tu PC; el servicio los usa solo y deja de cargar la cabeza propia.")

# Copia en partes de texto (xz + base64, 76.500 bytes por parte) para poder
# traerla desde el navegador: el visor de Kaggle muestra hasta 100 KB de un
# texto y nada de un binario.
man = {}
for nombre in ("horus_yolo4.pt", "horus_yolo4_ultimo.pt"):
    pt = pathlib.Path(f"{TRABAJO}/modelos_yolo/{nombre}")
    if not pt.exists() and nombre == "horus_yolo4.pt":
        pt = next(iter(sorted(pathlib.Path(f"{TRABAJO}/runs_yolo").rglob("best.pt"))), pt)
    if not pt.exists():
        print("NO está", nombre)
        continue
    d = pt.read_bytes()
    z = lzma.compress(d, preset=9 | lzma.PRESET_EXTREME)
    carpeta = "entrega/" + nombre.replace(".", "_")
    car = pathlib.Path(f"{TRABAJO}/{carpeta}")
    car.mkdir(parents=True, exist_ok=True)
    n = 0
    for i in range(0, len(z), 76_500):
        (car / f"{n:04d}.txt").write_text(base64.b64encode(z[i:i + 76_500]).decode())
        n += 1
    man[nombre] = {"origen": str(pt), "bytes": len(d), "sha256": hashlib.sha256(d).hexdigest(),
                   "xz_bytes": len(z), "xz_sha256": hashlib.sha256(z).hexdigest(),
                   "partes": n, "carpeta": carpeta}
pathlib.Path(f"{TRABAJO}/entrega").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{TRABAJO}/entrega/manifiesto.json").write_text(json.dumps(man, indent=1))
print(json.dumps(man, indent=1))

met = pathlib.Path(f"{TRABAJO}/modelos_yolo/metricas.json")
if met.exists():
    m = json.loads(met.read_text())
    # compacta (el visor de Kaggle corta en 100 KB)
    pathlib.Path(f"{TRABAJO}/modelos_yolo/metricas_min.json").write_text(
        json.dumps(m, separators=(",", ":"), ensure_ascii=False))
    for modelo, porset in m.items():
        for ds, r in porset.items():
            pf = r.get("por_foto", {})
            print(f"{modelo:>8} {ds:<13} AP50 " + " ".join(f"{k}={v:.2f}" for k, v in r.get("ap50", {}).items())
                  + " · @0.35 " + " ".join(f"{c}: ve {o['recall_0.35']:.0%} dispara {o['falsas_0.35']:.1%}"
                                           for c, o in pf.items() if o.get("fotos_con") or o.get("fotos_sin")))
for n in ("armas_ext", "fuego_ext"):
    p = pathlib.Path(f"{OBJ}/datasets/{n}/resumen.json")
    if p.exists():
        shutil.copy2(p, f"{TRABAJO}/modelos_yolo/{n}_resumen.json")